### Labels

In [1]:
# read csv with overlaps TU/e 
import pandas as pd 
filename = "/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/overlap_table.csv"
df_overlap = pd.read_csv(filename)
df_overlap[65:73]

,Course,New_code,Overlap
65,Security,2IRR40,2IC60
66,"Sets, Logic and Mathematic Language",2MBA10,2WF40
67,Statistics and Machine Learning,2IRR50,2IIG0 JBI030 2IIG5
68,Stochastic Optimization Models for Data Science,JBM100,2MBS30 2WB20
69,Stochastic Processes,2MBS30,2WB20 2WB29
70,Stochastic Simulation and Modeling,2MBS40,2WB50 2DI66
71,Stochastics and Simulation for Finance,2DF20,2WB50 2MBS40
72,Theory and Applications of Ordinary Differenti...,2MBC20,2WA70 2WA79


In [2]:
# manually input all old codes 
old_codes = ['2WN30', '2WF50', 'JBI040', '2WA30', 'JBM060', '2WF70', '2IT90', '2ILH0', 
             '2WA80', 'JBM020', 'JBM010', 'JBI025', 'JBI020', 'JBM090', 'JBM090-B-5', '2IAB0', '2WS30', '2WF60', 
             '2WF80', '2WN20', 'JBL120', '2WF20', '2WF20', '2WF30', '2DBI00', '2DRR00', 'JBM070', '2WO020', 
             '2WH60', '2WAG0', '2IC60', '2WN40', '2WA90', '2WS20', '2WH20', '2WB40', '2WS40', '2IC60', '2WF40', 
             '2IIG0', '2WB20', '2WB50', '2WA70', '2WA60'] 
print(len(old_codes))

44


In [3]:
# find unique ones 
all_overlap_codes = set()
for overlap_str in df_overlap['Overlap'].dropna():
    # Split by spaces to get individual codes
    codes = str(overlap_str).split()
    all_overlap_codes.update(codes)

# Step 2: Get all codes from New_code column
new_codes = set(df_overlap['New_code'].dropna().astype(str).str.strip())

# Step 3: Find overlap codes that are NOT in new_codes
codes_to_scrape = all_overlap_codes - new_codes

print(f"Step 3: Overlap codes NOT in New_code:")
print(f"Count: {len(codes_to_scrape)}")
#print(f"Codes: {sorted(codes_to_scrape)}")

Step 3: Overlap codes NOT in New_code:
Count: 99


In [4]:
all_old_codes = set() 
for old_code in old_codes: 
    all_old_codes.update(old_code)
print(len(all_old_codes))
# Step 3: Find overlap codes that are NOT in new_codes or an old code
overlap_codes = all_overlap_codes - new_codes - set(old_codes)
print(len(overlap_codes))
print(len(new_codes))

28
62
74


### Embed data 

In [5]:
# now we embed all course pages 
import nltk 
nltk.download('punkt') 
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize 
from sentence_transformers import SentenceTransformer 
import numpy as np 
import torch 

model_name = "sentence-transformers/all-mpnet-base-v2"   # sBERT model
model = SentenceTransformer(model_name)
tokenizer = model.tokenizer
max_tokens = 256  # model limit

import itertools
import re
from typing import List, Dict, Set
import pandas as pd 
from utils.embedding import * 
from utils.preprocessing import * 

[nltk_data] Downloading package punkt to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/NLP/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
[nltk_data] Downloading package punkt to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
from openai import OpenAI
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/'


json_files = find_json_files(output_path)
print('first json file: ',json_files[0])
parsed_json = load_json_file(json_files[0])
course_code = json_files[0].split('/')[-1].split('.')[0]  # '2IHA10'
print(course_code)
prompt = extract_course_info_from_json_prompt(parsed_json, course_code)
#print(prompt)
client = OpenAI(api_key ="sk-b14645cb792542729d8ac6b60626fef8", base_url="https://api.deepseek.com")

first json file:  /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2IHA10.json
2IHA10


In [8]:
new_codes_list = list(new_codes)
#overlap_codes = all_overlap_codes -  set(old_codes) # new codes can overlap with each other
overlap_codes_list = list(overlap_codes)
import ast 

all_courses = [] 
failed_courses = [] 
for course in new_codes_list: 
    try: 
        json_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/' + course + '.json' # find json filepath
        print(json_path)
        parsed_json = load_json_file(json_path) # parse json file 
        prompt = extract_course_info_from_json_prompt(parsed_json, course) # create prompt 
        response = client.chat.completions.create(
        model="deepseek-chat",
        messages = prompt, 
        stream = False) # pass prompt to DeepSeek 
        #print(response.choices[0].message.content) # print the output
        response_text = response.choices[0].message.content
        course_dict = ast.literal_eval(response_text) 
        print(course_dict['title'])
        course_dict['course_code'] = course 
        all_courses.append(course_dict) 
    except SyntaxError as e:
        print(f"✗ Syntax error for {course}: {e}")
        failed_courses.append(course)
        # Print the problematic response for debugging
        print(f"Problematic response: {response_text}")
        
    except Exception as e:
        print(f"✗ Unexpected error for {course}: {e}")
        failed_courses.append(course)
     

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI010.json
Programming
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0LVX10.json
Impact of technology engineering ethics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI026.json
Discrete Mathematics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA70.json
Measure and Probability Theory
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2ID50.json
Datamodeling and Databases
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA50.json
Linear Algebra 2
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBA30.json


KeyboardInterrupt: 

In [65]:
df_all_courses = pd.DataFrame(all_courses)

# Save to files
df_all_courses.to_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_CS_courses.csv', index=False, encoding='utf-8')
df_all_courses.to_pickle('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_CS_courses.pkl')

print(f"\n✓ Saved {len(all_courses)} courses successfully")
print(f"✗ Failed: {len(failed_courses)} courses")
print(f"\nDataFrame shape: {df_all_courses.shape}")
print(f"Columns: {df_all_courses.columns.tolist()}")
print("\nFirst row preview:")
print(df_all_courses.iloc[0])


✓ Saved 74 courses successfully
✗ Failed: 0 courses

DataFrame shape: (74, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
title                                     Measure and Probability Theory
credits                                                             5 EC
level                                                           Advanced
description            In this course, you will discover how measure ...
topics                 [Measures, measurable functions and sigma-alge...
learning_outcomes      [Define the Lebesgue integral and explain its ...
prerequisites          [2WA30 – Analysis 1, 2WA40 – Analysis 2, famil...
examination_methods                                [Written examination]
course_code                                                       2MBA70
Name: 0, dtype: object


In [66]:
from openai import OpenAI
new_codes_list = list(new_codes)
#overlap_codes = all_overlap_codes -  set(old_codes) # new codes can overlap with each other
overlap_codes_list = list(overlap_codes)
import ast 

all_courses = [] 
failed_courses = [] 
for course in overlap_codes_list: 
    try: 
        json_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/' + course + '.json' # find json filepath
        print(json_path)
        parsed_json = load_json_file(json_path) # parse json file 
        prompt = extract_course_info_from_json_prompt(parsed_json, course) # create prompt 
        response = client.chat.completions.create(
        model="deepseek-chat",
        messages = prompt, 
        stream = False) # pass prompt to DeepSeek 
        #print(response.choices[0].message.content) # print the output
        response_text = response.choices[0].message.content
        course_dict = ast.literal_eval(response_text) 
        print(course_dict['title'])
        course_dict['course_code'] = course 
        all_courses.append(course_dict) 
    except SyntaxError as e:
        print(f"✗ Syntax error for {course}: {e}")
        failed_courses.append(course)
        # Print the problematic response for debugging
        print(f"Problematic response: {response_text}")
        
    except Exception as e:
        print(f"✗ Unexpected error for {course}: {e}")
        failed_courses.append(course)

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0LVX10.json
Impact of technology engineering ethics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0HV60.json

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI025;.json

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2MBS40.json
Stochastic Simulation and Modeling
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2DBN10.json

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0LVX30.json
Impact of technology engineering ethics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/5EZB0.json

/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/0SV120.json
Statistics for Sustainable Innovation
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/JBI050.json
Data Management for Data Analytics
/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/2ITS60;.json

/mimer/NOBACKUP/groups/nais

In [67]:
df_all_courses = pd.DataFrame(all_courses)

# Save to files
df_all_courses.to_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_overlap_courses.csv', index=False, encoding='utf-8')
df_all_courses.to_pickle('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_overlap_courses.pkl')

print(f"\n✓ Saved {len(all_courses)} courses successfully")
print(f"✗ Failed: {len(failed_courses)} courses")
print(f"\nDataFrame shape: {df_all_courses.shape}")
print(f"Columns: {df_all_courses.columns.tolist()}")
print("\nFirst row preview:")
print(df_all_courses.iloc[0])


✓ Saved 91 courses successfully
✗ Failed: 1 courses

DataFrame shape: (91, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
title                            Impact of technology engineering ethics
credits                                                             5 EC
level                                                           Bachelor
description            Ethics of Technology and Engineering introduce...
topics                 [ethical aspects of engineering, normative ana...
learning_outcomes      [reflect on ethical aspects of engineering in ...
prerequisites          [Enrolled for one of: Applied Mathematics, App...
examination_methods    [Final Report (weight 70%, minimum grade 5.0),...
course_code                                                       0LVX10
Name: 0, dtype: object
